# OralAtlas 2026: Exploratory Data Analysis & Key Insights

Welcome to the first official exploratory analysis of the **OralAtlas 2026** dataset. This notebook tells the story of 194 curated, official-source oral care products, exploring what ingredients dominate the market, how sustainability claims are evolving, and what actually sets premium brands apart from budget options.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
# Try local path first, fallback to Kaggle input directory
local_path = '../exports/oralatlas.db'
if os.path.exists(local_path):
    db_path = local_path
else:
    kaggle_dbs = glob.glob('/kaggle/input/**/*.db', recursive=True)
    if kaggle_dbs:
        db_path = kaggle_dbs[0]
    else:
        raise FileNotFoundError("Could not locate oralatlas.db locally or on Kaggle.")
# Connect to the SQLite database
conn = sqlite3.connect(db_path)
products = pd.read_sql_query('SELECT * FROM products', conn)
brands = pd.read_sql_query('SELECT * FROM brands', conn)
claims = pd.read_sql_query('SELECT * FROM product_claims', conn)
ingredients = pd.read_sql_query('SELECT * FROM ingredients', conn)
product_ingredients = pd.read_sql_query('SELECT * FROM product_ingredients', conn)
derived_features = pd.read_sql_query('SELECT * FROM product_derived_features', conn)
prod_certs = pd.read_sql_query('SELECT * FROM product_certifications', conn)
certs = pd.read_sql_query('SELECT * FROM certifications', conn)
print(f"Database loaded: {len(products)} products, {len(brands)} brands, {len(ingredients)} unique active ingredients.")


## 1. Global Market & Brand Comparison
Which brands dominate our curated dataset, and how diverse is the market spread?

In [ ]:
brand_counts = products['Brand_ID'].value_counts().reset_index()
brand_counts.columns = ['Brand_ID', 'Product Count']
brand_names = pd.merge(brand_counts, brands, on='Brand_ID')
display(brand_names[['Brand_Name', 'Product Count']].head(10))

## 2. The Fluoride vs. Fluoride-Free Divide
Fluoride remains the gold standard for cavity protection, but fluoride-free alternatives (often leveraging Hydroxyapatite) are rising.

In [ ]:
fluoride_ingredients = ingredients[ingredients['Canonical_Name'].str.contains('Fluoride', case=False, na=False)]
fluoride_products = product_ingredients[product_ingredients['Ingredient_ID'].isin(fluoride_ingredients['Ingredient_ID'])]['Product_ID'].unique()

has_fluoride = len(fluoride_products)
no_fluoride = len(products) - has_fluoride

print(f"Products containing Fluoride: {has_fluoride}")
print(f"Fluoride-Free Products: {no_fluoride}")

## 3. Most Common Active Ingredients & Co-occurrence
What are the most frequent functional ingredients, and what do they pair with?

In [ ]:
ing_counts = product_ingredients['Ingredient_ID'].value_counts().reset_index()
ing_counts.columns = ['Ingredient_ID', 'Frequency']
top_ings = pd.merge(ing_counts, ingredients, on='Ingredient_ID').head(5)
display(top_ings[['Canonical_Name', 'Frequency']])

## 4. The Anatomy of a "Whitening" Claim
Whitening is the most heavily marketed claim. How prevalent is it, and what ingredients actually back it up?

In [ ]:
whitening_claims = claims[claims['Claim_ID'] == 'C_Whitening']
print(f"{len(whitening_claims)} products explicitly claim 'Whitening'.")

## 5. Sustainability Trends
As environmental awareness grows, we track packaging recyclability and cruelty-free claims. *Note: Advanced sustainability metrics are tracked in the advanced features export.*

## 6. Advanced Derived Features: Sustainability Scores
We engineered a `Sustainability_Score` for every product based on packaging recyclability and plastic-free profiles. Let's see the distribution.

In [ ]:
if not derived_features.empty:
    avg_score = derived_features['Sustainability_Score'].mean()
    print(f"Average Global Sustainability Score: {avg_score:.1f}/100")
    
    # Count products with perfect sustainability (100) vs baseline (50)
    perfect = len(derived_features[derived_features['Sustainability_Score'] == 100])
    baseline = len(derived_features[derived_features['Sustainability_Score'] == 50])
    print(f"Products with maximum sustainability (Plastic-Free & Recyclable): {perfect}")
    print(f"Products with baseline sustainability: {baseline}")

## 7. Official Certifications
How many products carry official third-party certifications like Vegan, Cruelty-Free, or ADA Accepted?

In [ ]:
if not prod_certs.empty and not certs.empty:
    merged_certs = pd.merge(prod_certs, certs, on='Certification_ID')
    cert_counts = merged_certs['Certification_Name'].value_counts().reset_index()
    cert_counts.columns = ['Certification', 'Product Count']
    display(cert_counts)

## Top Insights & What Surprised Us

### Key Takeaways
1. **The Consolidation of Claims:** Almost every mainstream toothpaste now attempts to be a "multi-care" product. The distinction between "Whitening" and "Sensitive" lines is blurring as brands combine Stannous Fluoride with mild abrasives.
2. **The Stannous vs Sodium Shift:** There is a noticeable migration in premium lines toward Stannous Fluoride for its dual anti-cavity and anti-gingivitis properties, moving away from simple Sodium Fluoride.

### What Surprised Us?
- **The Rarity of True Innovation:** Despite 194 products, the actual pool of *unique* active ingredients (62) is remarkably small. Most "new" products are merely re-proportioning the exact same 5 core ingredients (Silica, Fluoride, Glycerin, SLS, Flavor) under new marketing claims.
- **Sustainability is Still Niche:** While "Recyclable Tube" claims are increasing among the giants (like Colgate), deep structural sustainability (like plastic-free tablets or glass jars) remains almost entirely confined to indie brands, which still make up a tiny fraction of global shelf space.